In [1]:
import sys
import platform
import subprocess
from pathlib import Path

import os

HADOOP_DIR = r"C:\Additional\VirtualMachines\prac3\hadoop"
os.environ["HADOOP_HOME"] = HADOOP_DIR
os.environ["hadoop.home.dir"] = HADOOP_DIR
os.environ["PATH"] = os.path.join(HADOOP_DIR, "bin") + os.pathsep + os.environ["PATH"]

print(f"Python      : {sys.version.split()[0]}")

try:
    java_out = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True)
    print("Java        :", java_out.splitlines()[0])
except Exception as e:
    print("Java        : не удалось определить —", e)

import pyspark
print(f"PySpark     : {pyspark.__version__}")

Python      : 3.14.0
Java        : openjdk version "17.0.20.1" 2026-08-18
PySpark     : 4.2.0


In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

# Путь к исходной базе. Если файл лежит в подкаталоге — укажите точный путь,
# например: DB_PATH = Path(r"C:\Additional\VirtualMachines\prac3\ecommerce.sqlite")
DB_PATH = Path("ecommerce.sqlite")

STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

print("Текущий каталог :", Path.cwd())
print("SQLite существует:", DB_PATH.exists(), "->", DB_PATH.resolve())
print("Staging         :", STAGING.resolve())

Текущий каталог : C:\Additional\VirtualMachines\prac3
SQLite существует: True -> C:\Additional\VirtualMachines\prac3\ecommerce.sqlite
Staging         : C:\Additional\VirtualMachines\prac3\spark_input


In [3]:
TABLES = [
    "orders",
    "order_items",
    "products",
    "customers",
    "order_payments",
    "order_reviews",
    "product_category_name_translation",
]

with sqlite3.connect(DB_PATH) as conn:
    for table in TABLES:
        part = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
        part.to_csv(
            STAGING / f"{table}.csv",
            index=False,
            encoding="utf-8"
        )
        print(f"{table:36s} {len(part):>8} rows")

orders                                  99441 rows
order_items                            112650 rows
products                                32951 rows
customers                               99441 rows
order_payments                         103886 rows
order_reviews                           97621 rows
product_category_name_translation          71 rows


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Practice4_Ecommerce")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)

C:\Additional\VirtualMachines\prac3\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Master: local[*]
App: Practice4_Ecommerce
Spark: 4.2.0


In [5]:
orders_raw = (
 spark.read
 .option("header", True)
 .option("encoding", "UTF-8")
 .csv(str(STAGING / "orders.csv"))
)
orders_raw.printSchema()
print("Строк:", orders_raw.count())
orders_raw.select(
 "order_id",
 "customer_id",
 "order_status",
 "order_purchase_timestamp"
).show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

Строк: 99441
+--------------------------------+--------------------------------+------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|
+--------------------------------+--------------------------------+------------+------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |
|47770eb9100c2d0c44946d9cf07ec65d|41c

In [6]:
from pyspark.sql.types import (
 StructType, StructField,
 StringType, IntegerType, DoubleType
)
order_items_schema = StructType([
 StructField("order_id", StringType(), True),
 StructField("order_item_id", IntegerType(), True),
 StructField("product_id", StringType(), True),
 StructField("seller_id", StringType(), True),
 StructField("shipping_limit_date", StringType(), True),
 StructField("price", DoubleType(), True),
 StructField("freight_value", DoubleType(), True),
])
items = (
 spark.read
 .option("header", True)
 .schema(order_items_schema)
 .csv(str(STAGING / "order_items.csv"))
)
items.printSchema()
print("Строк:", items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Строк: 112650


In [7]:
timestamp_cols = [
 "order_purchase_timestamp",
 "order_approved_at",
 "order_delivered_carrier_date",
 "order_delivered_customer_date",
 "order_estimated_delivery_date",
]
orders = orders_raw
for c in timestamp_cols:
 orders = orders.withColumn(
 c,
 F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm:ss")
 )
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [8]:
expensive_items = (
 items
 .filter(F.col("price") >= 500)
 .withColumn(
 "gross_item_value",
 F.round(F.col("price") + F.col("freight_value"), 2)
 )
 .select(
 "order_id", "order_item_id",
 "price", "freight_value", "gross_item_value"
 )
)
print("Позиций с price >= 500:", expensive_items.count())
expensive_items.orderBy(
 F.desc("gross_item_value")
).show(5, truncate=False)


Позиций с price >= 500: 3239
+--------------------------------+-------------+------+-------------+----------------+
|order_id                        |order_item_id|price |freight_value|gross_item_value|
+--------------------------------+-------------+------+-------------+----------------+
|0812eb902a67711a1cb742b3cdaa65ae|1            |6735.0|194.31       |6929.31         |
|fefacc66af859508bf1a7934eab1e97f|1            |6729.0|193.21       |6922.21         |
|f5136e38d1a14a4dbd87dff67da82701|1            |6499.0|227.66       |6726.66         |
|a96610ab360d42a2e5335a3998b4718a|1            |4799.0|151.34       |4950.34         |
|199af31afc78c699f0dbf71fb178d4d4|1            |4690.0|74.34        |4764.34         |
+--------------------------------+-------------+------+-------------+----------------+
only showing top 5 rows


In [9]:
delivered = (
 orders
 .filter(F.col("order_status") == "delivered")
 .withColumn(
 "purchase_month",
 F.date_format("order_purchase_timestamp", "yyyy-MM")
 )
 .withColumn(
 "delivery_days",
 F.datediff(
 "order_delivered_customer_date",
 "order_purchase_timestamp"
 )
 )
 .withColumn(
 "is_late",
 F.col("order_delivered_customer_date") >
 F.col("order_estimated_delivery_date")
 )
)
with_delivery_date = delivered.filter(
 F.col("order_delivered_customer_date").isNotNull()
)
late = with_delivery_date.filter(F.col("is_late"))

print("Доставленных заказов:", delivered.count())
print("С известной датой доставки:", with_delivery_date.count())
print("Позже ожидаемой даты:", late.count())

Доставленных заказов: 96478
С известной датой доставки: 96470
Позже ожидаемой даты: 7826


In [10]:
null_counts = orders.select([
 F.sum(F.col(c).isNull().cast("int")).alias(c)
 for c in timestamp_cols
])
null_counts.show(vertical=True)

-RECORD 0-----------------------------
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



In [11]:
missing_delivery_by_status = (
 orders
 .filter(F.col("order_delivered_customer_date").isNull())
 .groupBy("order_status")
 .count()
 .orderBy(F.desc("count"))
)
missing_delivery_by_status.show(truncate=False)


+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |619  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|delivered   |8    |
|created     |5    |
|approved    |2    |
+------------+-----+



In [12]:
products = (
 spark.read
 .option("header", True)
 .option("inferSchema", True)
 .csv(str(STAGING / "products.csv"))
)
missing_categories = products.filter(
 F.col("product_category_name").isNull()
).count()
products_clean = products.fillna({
 "product_category_name": "unknown"
})
print("Товаров без категории:", missing_categories)

Товаров без категории: 610


In [13]:
status_stats = (
 orders
 .groupBy("order_status")
 .count()
 .orderBy(F.desc("count"))
)
status_stats.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



In [14]:
payments = (
 spark.read
 .option("header", True)
 .option("inferSchema", True)
 .csv(str(STAGING / "order_payments.csv"))
)
payment_stats = (
 payments
 .groupBy("payment_type")
 .agg(
 F.count("*").alias("payments"),
 F.round(F.sum("payment_value"), 2).alias("total_value"),
 F.round(F.avg("payment_value"), 2).alias("avg_value")
 )
 .orderBy(F.desc("total_value"))
)
payment_stats.show(truncate=False)

+------------+--------+-------------+---------+
|payment_type|payments|total_value  |avg_value|
+------------+--------+-------------+---------+
|credit_card |76795   |1.254208419E7|163.32   |
|boleto      |19784   |2869361.27   |145.03   |
|voucher     |5775    |379436.87    |65.7     |
|debit_card  |1529    |217989.79    |142.57   |
|not_defined |3       |0.0          |0.0      |
+------------+--------+-------------+---------+



In [15]:
translations = (
 spark.read
 .option("header", True)
 .csv(str(STAGING / "product_category_name_translation.csv"))
)
sales = (
 items.alias("i")
 .join(
 products_clean.select(
 "product_id", "product_category_name"
 ).alias("p"),
 F.col("i.product_id") == F.col("p.product_id"),
 "left"
 )
 .join(
 translations.alias("t"),
 F.col("p.product_category_name") ==
 F.col("t.product_category_name"),
 "left"
 )
 .select(
 F.col("i.order_id").alias("order_id"),
 F.col("i.order_item_id").alias("order_item_id"),
 F.col("i.product_id").alias("product_id"),
 F.col("i.price").alias("price"),
 F.col("i.freight_value").alias("freight_value"),
 F.coalesce(
 F.col("t.product_category_name_english"),
 F.col("p.product_category_name"),
 F.lit("unknown")
 ).alias("category")
 )
)
print("Строк до JOIN:", items.count())
print("Строк после JOIN:", sales.count())
print(
 "Позиций категории unknown:",
 sales.filter(F.col("category") == "unknown").count()
)

Строк до JOIN: 112650
Строк после JOIN: 112650
Позиций категории unknown: 1603


In [16]:
customers = (
 spark.read
 .option("header", True)
 .option("inferSchema", True)
 .csv(str(STAGING / "customers.csv"))
)
late_by_state = (
 delivered
 .join(
 customers.select("customer_id", "customer_state"),
 on="customer_id",
 how="left"
 )
 .filter(F.col("order_delivered_customer_date").isNotNull())
 .groupBy("customer_state")
 .agg(
 F.count("*").alias("delivered"),
 F.sum(
 F.when(F.col("is_late"), 1).otherwise(0)
 ).alias("late")
 )
 .withColumn(
 "late_pct",
 F.round(F.col("late") / F.col("delivered") * 100, 2)
 )
 .orderBy(F.desc("late_pct"))
)
late_by_state.show(10, truncate=False)

+--------------+---------+----+--------+
|customer_state|delivered|late|late_pct|
+--------------+---------+----+--------+
|AL            |397      |95  |23.93   |
|MA            |717      |141 |19.67   |
|PI            |476      |76  |15.97   |
|CE            |1279     |196 |15.32   |
|SE            |335      |51  |15.22   |
|BA            |3256     |457 |14.04   |
|RJ            |12350    |1664|13.47   |
|TO            |274      |35  |12.77   |
|PA            |946      |117 |12.37   |
|ES            |1995     |244 |12.23   |
+--------------+---------+----+--------+
only showing top 10 rows


In [17]:
sales.createOrReplaceTempView("sales")
category_sql = spark.sql("""
SELECT
 category,
 COUNT(*) AS items,
 COUNT(DISTINCT order_id) AS orders,
 ROUND(SUM(price), 2) AS revenue,
 ROUND(AVG(price), 2) AS avg_price
FROM sales
GROUP BY category
ORDER BY revenue DESC
LIMIT 10
""")
category_sql.show(truncate=False)


+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+



In [18]:
category_sales = (
 sales
 .groupBy("category")
 .agg(
 F.count("*").alias("items"),
 F.countDistinct("order_id").alias("orders"),
 F.round(F.sum("price"), 2).alias("revenue"),
 F.round(F.avg("price"), 2).alias("avg_price")
 )
)
print("Категорий в витрине:", category_sales.count())
category_sales.orderBy(
 F.desc("revenue")
).show(10, truncate=False)


Категорий в витрине: 74
+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+
only showing top 10 rows


In [19]:
import os

os.environ["HADOOP_HOME"] = r"C:\Additional\VirtualMachines\prac3\hadoop"
os.environ["hadoop.home.dir"] = r"C:\Additional\VirtualMachines\prac3\hadoop"

OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)
(
 category_sales
 .write
 .mode("overwrite")
 .parquet(str(OUT / "category_sales_parquet"))
)
(
 category_sales
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(str(OUT / "category_sales_csv"))
)
category_back = spark.read.parquet(
 str(OUT / "category_sales_parquet")
)
print("Строк после round-trip:", category_back.count())
category_back.printSchema()

Строк после round-trip: 74
root
 |-- category: string (nullable = true)
 |-- items: long (nullable = true)
 |-- orders: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- avg_price: double (nullable = true)



In [20]:
spark.stop()